### CROMA embeddings per FID

In [ ]:
import os
import glob
import numpy as np
import torch
import rasterio
import matplotlib.pyplot as plt
from torchgeo.models import croma_base, CROMABase_Weights

TILES_DIR = os.path.expanduser("~/thesis_tiles_120px")

PATCH_SIZE = 8
EMBED_DIM = 768
TILE_SIZE = 120
GRID_SIZE = TILE_SIZE // PATCH_SIZE     # 15
CROMA_S2_BAND_INDICES = list(range(12))

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
print(f"Device: {device}")
print(f"Tiles dir: {TILES_DIR}")

#find the tiles in the folder
def find_all_tiles(fid, product, window, image_id):
    pattern = os.path.join(TILES_DIR, product, f"fid_{fid}", window, image_id, "tile_*.tif")
    return sorted(glob.glob(pattern))

#images names
def list_image_ids(fid, product, window):
    win_dir = os.path.join(TILES_DIR, product, f"fid_{fid}", window)
    if not os.path.isdir(win_dir):
        return []
    return sorted(os.listdir(win_dir))


In [ ]:
#load one tile
def load_tile(path, band_indices=None):
    with rasterio.open(path) as src:
        data = src.read().astype(np.float32)
    if band_indices is not None:
        data = data[band_indices]
    return data


def normalize(x):
    """CROMA recipe: μ ± 2σ per channel over the input batch x (N, C, H, W)."""
    x = x.float()
    imgs = []
    for ch in range(x.shape[1]):
        channel = x[:, ch, :, :]
        min_val = channel.mean() - 2 * channel.std()
        max_val = channel.mean() + 2 * channel.std()
        img = (channel - min_val) / (max_val - min_val + 1e-10)
        img = torch.clamp(img, 0, 1)
        imgs.append(img.unsqueeze(1))
    return torch.cat(imgs, dim=1)


def encode(model, x_normalized, modality):
    """Forward pass on a normalized batch. Returns (N, 15, 15, 768) tokens."""
    captured = {}
    encoder = model.s2_encoder if modality == "optical" else model.s1_encoder

    def hook(module, inp, out):
        captured["tokens"] = out

    h = encoder.register_forward_hook(hook)
    with torch.no_grad():
        if modality == "optical":
            _ = model(x_optical=x_normalized)
        else:
            _ = model(x_sar=x_normalized)
    h.remove()

    out = captured["tokens"]
    if isinstance(out, tuple):
        out = out[0]
    tokens = out.cpu().numpy()
    if tokens.ndim == 2:
        tokens = tokens[None]
    expected = GRID_SIZE * GRID_SIZE
    if tokens.shape[1] == expected + 1:
        tokens = tokens[:, 1:]
    return tokens.reshape(-1, GRID_SIZE, GRID_SIZE, EMBED_DIM)


In [ ]:
model = croma_base(
    weights=CROMABase_Weights.CROMA_VIT,
    modalities=["optical"],
    image_size=TILE_SIZE,
).to(device).eval()
print("CROMA optical model loaded")


In [ ]:
FID = "112"           # adjust to any FID you have on disk
WINDOW = "bef"        # bef / evt / aft
PRODUCT = "s2_l2a"
MODALITY = "optical"

# 1) Pick first available acquisition for this (FID, window)
image_ids = list_image_ids(FID, PRODUCT, WINDOW)
assert image_ids, f"No image_ids for fid={FID} {WINDOW}"
image_id = image_ids[0]
paths = find_all_tiles(FID, PRODUCT, WINDOW, image_id)
print(f"FID={FID}  window={WINDOW}  image_id={image_id}")
print(f"Tiles found: {len(paths)}")

# 2) Load all tiles and stack into a batch (N, C, H, W)
# N = number of tiles
# C = number of bands
# H = height
# W = width
all_data = np.stack(
    [load_tile(p, band_indices=CROMA_S2_BAND_INDICES) for p in paths],
    axis=0,
).astype(np.float32)
print(f"Batch shape: {all_data.shape}")

# 3) Normalize jointly (stats over the N tiles) and encode
x = torch.from_numpy(all_data)
x = normalize(x).to(device)
tokens = encode(model, x, MODALITY)        # (N, 15, 15, 768)
print(f"Tokens shape: {tokens.shape}")


In [ ]:
# Print values of ONE embedding dimension across all patches of all tiles.
DIM = 0   # pick any dimension in [0, 767]

print(f"Values of dimension {DIM} for each tile (15x15 grid of patches):\n")
for i, t in enumerate(tokens):
    print(f"--- tile {i} (path: {os.path.basename(paths[i])}) ---")
    # t has shape (15, 15, 768); slice the chosen dimension
    plane = t[:, :, DIM]
    np.set_printoptions(precision=3, suppress=True, linewidth=160)
    print(plane)
    print()


In [ ]:
# Stitch tile tokens into a mosaic using each tile's geo-coordinates.
TILE_SIZE_M = TILE_SIZE * 10   # 1200 m per tile

transforms = []
for p in paths:
    with rasterio.open(p) as src:
        transforms.append(src.transform)

cs = [t.c for t in transforms]
fs = [t.f for t in transforms]
x0 = min(cs)
y0 = max(fs)
n_cols = int(round((max(cs) - x0) / TILE_SIZE_M)) + 1
n_rows = int(round((y0 - min(fs)) / TILE_SIZE_M)) + 1

mosaic = np.full((n_rows * GRID_SIZE, n_cols * GRID_SIZE, EMBED_DIM), np.nan, dtype=np.float32)
for i, p in enumerate(paths):
    with rasterio.open(p) as src:
        t = src.transform
    row = int(round((y0 - t.f) / TILE_SIZE_M))
    col = int(round((t.c - x0) / TILE_SIZE_M))
    mosaic[row * GRID_SIZE:(row + 1) * GRID_SIZE,
           col * GRID_SIZE:(col + 1) * GRID_SIZE, :] = tokens[i]

print(f"Mosaic: {n_rows}x{n_cols} tiles -> token grid {mosaic.shape}")

# Global min/max per embedding dimension across ALL valid patches in the mosaic
flat = mosaic.reshape(-1, EMBED_DIM)
valid_mask = ~np.isnan(flat).any(axis=1)
d_min = flat[valid_mask].min(axis=0)     # shape (768,)
d_max = flat[valid_mask].max(axis=0)     # shape (768,)
print(f"Per-dim min range: [{d_min.min():.3f}, {d_min.max():.3f}]")
print(f"Per-dim max range: [{d_max.min():.3f}, {d_max.max():.3f}]")

# Min-max normalize: I_norm = (I - I_min) / (I_max - I_min)
mosaic_norm = (mosaic - d_min) / (d_max - d_min + 1e-10)

# Plot the first 3 dimensions as RGB
rgb = mosaic_norm[:, :, :3]
rgb_disp = np.where(np.isnan(rgb), 0.5, rgb)

plt.figure(figsize=(7, 7))
plt.imshow(rgb_disp, interpolation="nearest")
plt.title(f"FID {FID} {WINDOW} — dims 0/1/2 as RGB (min-max per dim)")
plt.axis("off")
plt.show()
